# 📈 Notebook 3: Differential Equation Modeling

## DE/IDE Feature Extraction with Euler & RK4 Methods

---

### Pipeline Stage
```
                                    ┌────────────────────────────────┐
  1D Signal (10,)  ────▶  │  Differential Modeling         │
  from Notebook 02        │  ─ ODE: dy/dt = f(y, t)        │
                          │  ─ IDE: dy/dt = f + ∫K(t,s)ds │
                          │  ─ Solvers: Euler & RK4        │
                          └────────────────────────────────┘
                                      │
                                      ▼
                          Feature Vector → CNN (Notebook 04)
```

### Objective
Model the preprocessed 1D signals using **Differential Equations (DE)** and **Integro-Differential Equations (IDE)**. Extract discriminative features using both **Euler's method** and **Runge-Kutta 4th order (RK4)** numerical solvers.

### Mathematical Background

#### Ordinary Differential Equation (ODE)
We model the 1D signal $y(t)$ as governed by:
$$\frac{dy}{dt} = f(y, t) = \alpha y + \beta t + \gamma$$

#### Integro-Differential Equation (IDE)
$$\frac{dy}{dt} = f(y, t) + \lambda \int_0^t K(t, s) \cdot y(s) \, ds$$
where $K(t, s) = e^{-\mu(t-s)}$ is an exponential decay kernel.

## 1. Import Dependencies

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✅ All imports successful")

## 2. Load Preprocessed Data

In [ ]:
OUTPUT_DIR = './processed_data'

# Load preprocessed signals from Notebook 02
data_path = os.path.join(OUTPUT_DIR, 'preprocessed_signals.npz')

if os.path.exists(data_path):
    data = np.load(data_path)
    signals = data['signals']
    labels = data['labels']
    print(f"✅ Loaded preprocessed data from Notebook 02")
else:
    # Generate synthetic signals for demo
    print("⚠️ Preprocessed data not found. Generating synthetic signals...")
    n_samples = 1000
    signals = np.zeros((n_samples, 10), dtype=np.float32)
    labels = np.zeros(n_samples, dtype=np.int32)
    
    for i in range(n_samples):
        is_malignant = np.random.random() < 0.6
        labels[i] = 1 if is_malignant else 0
        
        t = np.linspace(0, 1, 10)
        if is_malignant:
            # Malignant: higher amplitude, more irregular
            signals[i] = 0.3 + 0.4 * np.sin(2 * np.pi * t * np.random.uniform(0.8, 1.5)) + \
                         np.random.normal(0, 0.06, 10)
        else:
            # Benign: smoother, lower amplitude
            signals[i] = 0.25 + 0.2 * np.sin(2 * np.pi * t * np.random.uniform(0.5, 1.0)) + \
                         np.random.normal(0, 0.03, 10)
    
    signals = np.clip(signals, 0, 1)

print(f"   Signals shape : {signals.shape}")
print(f"   Labels shape  : {labels.shape}")
print(f"   Malignant     : {np.sum(labels == 1)}")
print(f"   Benign        : {np.sum(labels == 0)}")

## 3. Numerical Differentiation

Before fitting a DE model, we compute the numerical derivative of each signal using finite differences:

$$\left.\frac{dy}{dt}\right|_{t_i} \approx \frac{y(t_{i+1}) - y(t_{i-1})}{2 \Delta t}$$

(Central difference scheme for interior points, forward/backward for boundaries)

In [ ]:
def numerical_derivative(signal, dt=1.0):
    """
    Compute the numerical derivative using central differences.
    
    Parameters:
        signal: 1D array of length N
        dt: time step (default 1.0)
    
    Returns:
        derivative: 1D array of length N
    """
    N = len(signal)
    deriv = np.zeros(N)
    
    # Forward difference at start
    deriv[0] = (signal[1] - signal[0]) / dt
    
    # Central differences for interior points
    for i in range(1, N - 1):
        deriv[i] = (signal[i + 1] - signal[i - 1]) / (2 * dt)
    
    # Backward difference at end
    deriv[N - 1] = (signal[N - 1] - signal[N - 2]) / dt
    
    return deriv


# Demo: compute derivative for a sample signal
sample_sig = signals[0]
sample_deriv = numerical_derivative(sample_sig)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
t = np.arange(10)

ax1.plot(t, sample_sig, 'o-', color='steelblue', linewidth=2, markersize=8, label='y(t)')
ax1.set_title('Original Signal y(t)', fontsize=13, fontweight='bold')
ax1.set_xlabel('t (row index)')
ax1.set_ylabel('y(t)')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

ax2.plot(t, sample_deriv, 's-', color='darkorange', linewidth=2, markersize=8, label="dy/dt")
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Numerical Derivative dy/dt', fontsize=13, fontweight='bold')
ax2.set_xlabel('t (row index)')
ax2.set_ylabel('dy/dt')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

## 4. ODE Model Fitting

We fit a parametric ODE model to each signal. The model assumes:

$$\frac{dy}{dt} = \alpha \cdot y + \beta \cdot t + \gamma$$

The parameters $(\alpha, \beta, \gamma)$ are estimated via least squares regression using the observed values of $y(t)$ and the numerically computed $\frac{dy}{dt}$.

In [ ]:
def fit_ode_model(signal, dt=1.0):
    """
    Fit an ODE model: dy/dt = alpha * y + beta * t + gamma
    
    Parameters:
        signal: 1D array of length N
        dt: time step
    
    Returns:
        params: dict with 'alpha', 'beta', 'gamma' coefficients
        residual: fitting residual (RMSE)
    """
    N = len(signal)
    t = np.arange(N, dtype=np.float64) * dt
    y = signal.astype(np.float64)
    dydt = numerical_derivative(y, dt)
    
    # Set up least squares: dydt = alpha * y + beta * t + gamma
    # Design matrix: [y, t, 1]
    A = np.column_stack([y, t, np.ones(N)])
    
    # Solve via least squares
    result = np.linalg.lstsq(A, dydt, rcond=None)
    coefficients = result[0]
    
    alpha, beta, gamma = coefficients
    
    # Compute residual
    dydt_pred = alpha * y + beta * t + gamma
    residual = np.sqrt(np.mean((dydt - dydt_pred) ** 2))
    
    return {
        'alpha': alpha,
        'beta': beta,
        'gamma': gamma
    }, residual


# Demo fit
params, res = fit_ode_model(sample_sig)
print(f"ODE Model fitted to sample signal:")
print(f"   dy/dt = {params['alpha']:.4f} * y + {params['beta']:.4f} * t + {params['gamma']:.4f}")
print(f"   Fitting RMSE: {res:.6f}")

## 5. Euler's Method

**Euler's method** is a first-order numerical method for solving initial value problems:

$$y_{n+1} = y_n + h \cdot f(y_n, t_n)$$

where $h = \Delta t$ is the step size and $f(y, t) = \alpha y + \beta t + \gamma$ is our fitted ODE.

In [ ]:
def euler_method(f, y0, t_span, h):
    """
    Solve ODE dy/dt = f(y, t) using Euler's method.
    
    Parameters:
        f: function f(y, t) returning dy/dt
        y0: initial condition y(t0)
        t_span: array of time points
        h: step size
    
    Returns:
        y: solution array of same length as t_span
    """
    N = len(t_span)
    y = np.zeros(N)
    y[0] = y0
    
    for i in range(N - 1):
        y[i + 1] = y[i] + h * f(y[i], t_span[i])
    
    return y


def solve_with_euler(signal, params, dt=1.0):
    """
    Solve the fitted ODE using Euler's method.
    
    Returns:
        euler_trajectory: predicted signal from Euler's method
    """
    alpha, beta, gamma = params['alpha'], params['beta'], params['gamma']
    
    def f(y, t):
        return alpha * y + beta * t + gamma
    
    N = len(signal)
    t_span = np.arange(N, dtype=np.float64) * dt
    y0 = signal[0]
    
    return euler_method(f, y0, t_span, dt)


# Demo
euler_traj = solve_with_euler(sample_sig, params)
print(f"Euler's Method Solution:")
print(f"   Trajectory: {np.round(euler_traj, 4)}")
print(f"   RMSE vs original: {np.sqrt(np.mean((sample_sig - euler_traj)**2)):.6f}")

## 6. Runge-Kutta 4th Order (RK4) Method

**RK4** is a higher-accuracy numerical method that evaluates the derivative at four points per step:

$$\begin{align}
k_1 &= h \cdot f(y_n, t_n) \\
k_2 &= h \cdot f\left(y_n + \frac{k_1}{2}, t_n + \frac{h}{2}\right) \\
k_3 &= h \cdot f\left(y_n + \frac{k_2}{2}, t_n + \frac{h}{2}\right) \\
k_4 &= h \cdot f(y_n + k_3, t_n + h) \\
y_{n+1} &= y_n + \frac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4)
\end{align}$$

In [ ]:
def rk4_method(f, y0, t_span, h):
    """
    Solve ODE dy/dt = f(y, t) using the 4th-order Runge-Kutta method.
    
    Parameters:
        f: function f(y, t) returning dy/dt
        y0: initial condition y(t0)
        t_span: array of time points
        h: step size
    
    Returns:
        y: solution array of same length as t_span
    """
    N = len(t_span)
    y = np.zeros(N)
    y[0] = y0
    
    for i in range(N - 1):
        k1 = h * f(y[i], t_span[i])
        k2 = h * f(y[i] + k1 / 2, t_span[i] + h / 2)
        k3 = h * f(y[i] + k2 / 2, t_span[i] + h / 2)
        k4 = h * f(y[i] + k3, t_span[i] + h)
        
        y[i + 1] = y[i] + (k1 + 2 * k2 + 2 * k3 + k4) / 6
    
    return y


def solve_with_rk4(signal, params, dt=1.0):
    """
    Solve the fitted ODE using the RK4 method.
    
    Returns:
        rk4_trajectory: predicted signal from RK4 method
    """
    alpha, beta, gamma = params['alpha'], params['beta'], params['gamma']
    
    def f(y, t):
        return alpha * y + beta * t + gamma
    
    N = len(signal)
    t_span = np.arange(N, dtype=np.float64) * dt
    y0 = signal[0]
    
    return rk4_method(f, y0, t_span, dt)


# Demo
rk4_traj = solve_with_rk4(sample_sig, params)
print(f"RK4 Method Solution:")
print(f"   Trajectory: {np.round(rk4_traj, 4)}")
print(f"   RMSE vs original: {np.sqrt(np.mean((sample_sig - rk4_traj)**2)):.6f}")

## 7. Integro-Differential Equation (IDE) Modeling

The IDE extends the ODE by incorporating a **memory integral term**:

$$\frac{dy}{dt} = f(y, t) + \lambda \int_0^t K(t, s) \cdot y(s) \, ds$$

where:
- $K(t, s) = e^{-\mu(t - s)}$ is an **exponential decay kernel** (recent values contribute more)
- $\lambda$ controls the **influence strength** of the integral term
- $\mu$ controls the **decay rate** of the memory kernel

In [ ]:
def compute_integral_term(signal, t_idx, mu=0.5, dt=1.0):
    """
    Compute the integral term: ∫₀ᵗ K(t,s) * y(s) ds
    where K(t,s) = exp(-mu * (t - s))
    
    Uses trapezoidal rule for numerical integration.
    """
    if t_idx == 0:
        return 0.0
    
    s_values = np.arange(t_idx + 1, dtype=np.float64) * dt
    t_val = t_idx * dt
    
    # Kernel values: K(t, s) = exp(-mu * (t - s))
    kernel_vals = np.exp(-mu * (t_val - s_values))
    
    # Integrand: K(t, s) * y(s)
    integrand = kernel_vals * signal[:t_idx + 1]
    
    # Trapezoidal integration
    integral = np.trapz(integrand, s_values)
    
    return integral


def euler_ide(signal, params, lam=0.1, mu=0.5, dt=1.0):
    """
    Solve the IDE using Euler's method.
    dy/dt = alpha*y + beta*t + gamma + lambda * integral(K(t,s)*y(s), 0, t)
    """
    alpha, beta, gamma = params['alpha'], params['beta'], params['gamma']
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    
    for i in range(N - 1):
        t_val = i * dt
        ode_term = alpha * y[i] + beta * t_val + gamma
        integral_term = lam * compute_integral_term(y, i, mu, dt)
        
        y[i + 1] = y[i] + dt * (ode_term + integral_term)
    
    return y


def rk4_ide(signal, params, lam=0.1, mu=0.5, dt=1.0):
    """
    Solve the IDE using RK4 method (simplified - uses Euler for integral part).
    The integral term is computed based on the solution up to the current step.
    """
    alpha, beta, gamma = params['alpha'], params['beta'], params['gamma']
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    
    for i in range(N - 1):
        t_val = i * dt
        int_term = lam * compute_integral_term(y, i, mu, dt)
        
        def f(y_val, t_val_local):
            return alpha * y_val + beta * t_val_local + gamma + int_term
        
        k1 = dt * f(y[i], t_val)
        k2 = dt * f(y[i] + k1/2, t_val + dt/2)
        k3 = dt * f(y[i] + k2/2, t_val + dt/2)
        k4 = dt * f(y[i] + k3, t_val + dt)
        
        y[i + 1] = y[i] + (k1 + 2*k2 + 2*k3 + k4) / 6
    
    return y


# Demo
euler_ide_traj = euler_ide(sample_sig, params, lam=0.05, mu=0.3)
rk4_ide_traj = rk4_ide(sample_sig, params, lam=0.05, mu=0.3)

print("IDE Solutions computed:")
print(f"   Euler IDE RMSE: {np.sqrt(np.mean((sample_sig - euler_ide_traj)**2)):.6f}")
print(f"   RK4 IDE RMSE  : {np.sqrt(np.mean((sample_sig - rk4_ide_traj)**2)):.6f}")

## 8. Visualize All Solutions for Sample Signal

Compare the original signal with all four numerical solutions:
- Euler ODE
- RK4 ODE
- Euler IDE
- RK4 IDE

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
t = np.arange(10)

solutions = [
    ('Euler ODE', euler_traj, 'darkorange'),
    ('RK4 ODE', rk4_traj, 'forestgreen'),
    ('Euler IDE', euler_ide_traj, 'mediumpurple'),
    ('RK4 IDE', rk4_ide_traj, 'crimson')
]

for idx, (name, traj, color) in enumerate(solutions):
    ax = axes[idx // 2, idx % 2]
    
    # Original signal
    ax.plot(t, sample_sig, 'ko-', linewidth=2, markersize=8, label='Original y(t)', zorder=5)
    
    # Solution trajectory
    ax.plot(t, traj, 's--', color=color, linewidth=2, markersize=7, label=f'{name}', alpha=0.8)
    
    # Residuals (error bars)
    residuals = sample_sig - traj
    for i in range(len(t)):
        ax.vlines(t[i], min(sample_sig[i], traj[i]), max(sample_sig[i], traj[i]),
                  color=color, alpha=0.3, linewidth=3)
    
    rmse = np.sqrt(np.mean(residuals ** 2))
    ax.set_title(f'{name}  |  RMSE = {rmse:.4f}', fontsize=13, fontweight='bold')
    ax.set_xlabel('t (row index)')
    ax.set_ylabel('Signal Value')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)

fig.suptitle('Comparison of Numerical Solutions vs Original Signal', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'de_solutions_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Extraction Pipeline

For each signal, we extract a comprehensive feature vector by combining outputs from all four solving methods.

### Feature Vector Composition

| Feature Group | Size | Description |
|--------------|------|-------------|
| ODE Coefficients | 3 | α, β, γ from fitted ODE model |
| Euler ODE Trajectory | 10 | Predicted signal from Euler's method |
| RK4 ODE Trajectory | 10 | Predicted signal from RK4 method |
| Euler ODE Residuals | 10 | Original - Euler predicted |
| RK4 ODE Residuals | 10 | Original - RK4 predicted |
| Euler IDE Trajectory | 10 | IDE solution via Euler |
| RK4 IDE Trajectory | 10 | IDE solution via RK4 |
| Euler IDE Residuals | 10 | Original - Euler IDE |
| RK4 IDE Residuals | 10 | Original - RK4 IDE |
| Statistical Features | 7 | RMSE, energy, slope, curvature, etc. |
| **Total** | **90** | Complete differential feature vector |

In [ ]:
def extract_de_features(signal, dt=1.0, lam=0.05, mu=0.3):
    """
    Extract comprehensive differential equation features from a 1D signal.
    
    Parameters:
        signal: 1D array of length N (the preprocessed row-mean signal)
        dt: time step
        lam: IDE integral weight
        mu: IDE kernel decay rate
    
    Returns:
        features: 1D feature vector
        feature_names: list of feature name strings
    """
    N = len(signal)
    signal = signal.astype(np.float64)
    
    # --- Step 1: Fit ODE model ---
    params, fit_rmse = fit_ode_model(signal, dt)
    alpha, beta, gamma = params['alpha'], params['beta'], params['gamma']
    
    # --- Step 2: Solve with Euler (ODE) ---
    euler_ode_traj = solve_with_euler(signal, params, dt)
    euler_ode_res = signal - euler_ode_traj
    
    # --- Step 3: Solve with RK4 (ODE) ---
    rk4_ode_traj = solve_with_rk4(signal, params, dt)
    rk4_ode_res = signal - rk4_ode_traj
    
    # --- Step 4: Solve with Euler (IDE) ---
    euler_ide_traj_sol = euler_ide(signal, params, lam, mu, dt)
    euler_ide_res = signal - euler_ide_traj_sol
    
    # --- Step 5: Solve with RK4 (IDE) ---
    rk4_ide_traj_sol = rk4_ide(signal, params, lam, mu, dt)
    rk4_ide_res = signal - rk4_ide_traj_sol
    
    # --- Step 6: Statistical features ---
    euler_ode_rmse = np.sqrt(np.mean(euler_ode_res ** 2))
    rk4_ode_rmse = np.sqrt(np.mean(rk4_ode_res ** 2))
    euler_ide_rmse = np.sqrt(np.mean(euler_ide_res ** 2))
    rk4_ide_rmse = np.sqrt(np.mean(rk4_ide_res ** 2))
    signal_energy = np.sum(signal ** 2)
    signal_slope = signal[-1] - signal[0]  # Overall trend
    signal_curvature = np.mean(np.abs(np.diff(signal, n=2)))  # Avg curvature
    
    # --- Assemble feature vector ---
    features = np.concatenate([
        # ODE coefficients (3)
        [alpha, beta, gamma],
        # Euler ODE trajectory (10)
        euler_ode_traj,
        # RK4 ODE trajectory (10)
        rk4_ode_traj,
        # Euler ODE residuals (10)
        euler_ode_res,
        # RK4 ODE residuals (10)
        rk4_ode_res,
        # Euler IDE trajectory (10)
        euler_ide_traj_sol,
        # RK4 IDE trajectory (10)
        rk4_ide_traj_sol,
        # Euler IDE residuals (10)
        euler_ide_res,
        # RK4 IDE residuals (10)
        rk4_ide_res,
        # Statistical features (7)
        [euler_ode_rmse, rk4_ode_rmse, euler_ide_rmse, rk4_ide_rmse,
         signal_energy, signal_slope, signal_curvature]
    ])
    
    # Feature names
    feature_names = (
        ['alpha', 'beta', 'gamma'] +
        [f'euler_ode_traj_{i}' for i in range(N)] +
        [f'rk4_ode_traj_{i}' for i in range(N)] +
        [f'euler_ode_res_{i}' for i in range(N)] +
        [f'rk4_ode_res_{i}' for i in range(N)] +
        [f'euler_ide_traj_{i}' for i in range(N)] +
        [f'rk4_ide_traj_{i}' for i in range(N)] +
        [f'euler_ide_res_{i}' for i in range(N)] +
        [f'rk4_ide_res_{i}' for i in range(N)] +
        ['euler_ode_rmse', 'rk4_ode_rmse', 'euler_ide_rmse', 'rk4_ide_rmse',
         'signal_energy', 'signal_slope', 'signal_curvature']
    )
    
    return features, feature_names


# Demo
feat, feat_names = extract_de_features(sample_sig)
print(f"Feature vector length: {len(feat)}")
print(f"Feature names (first 10): {feat_names[:10]}")
print(f"Feature values (first 10): {np.round(feat[:10], 4)}")

## 10. Process Entire Dataset

Apply the DE/IDE feature extraction to ALL preprocessed signals.

In [ ]:
print("\n" + "="*60)
print(" EXTRACTING DIFFERENTIAL EQUATION FEATURES")
print("="*60)

all_features = []
feature_names = None
failed_count = 0

for i in tqdm(range(len(signals)), desc="Extracting DE features"):
    try:
        feat, names = extract_de_features(signals[i])
        all_features.append(feat)
        if feature_names is None:
            feature_names = names
    except Exception as e:
        # Fallback: zero features for failed samples
        if feature_names is not None:
            all_features.append(np.zeros(len(feature_names)))
        else:
            all_features.append(np.zeros(90))
        failed_count += 1

features_matrix = np.array(all_features, dtype=np.float32)

# Handle any NaN/Inf values
features_matrix = np.nan_to_num(features_matrix, nan=0.0, posinf=1.0, neginf=-1.0)

print(f"\n📊 Feature Extraction Results:")
print(f"   Feature matrix shape   : {features_matrix.shape}")
print(f"   Number of features     : {features_matrix.shape[1]}")
print(f"   Failed extractions     : {failed_count}")
print(f"   Feature range          : [{features_matrix.min():.4f}, {features_matrix.max():.4f}]")
print(f"   Feature mean           : {features_matrix.mean():.4f}")
print(f"   Features with NaN      : {np.sum(np.isnan(features_matrix))}")

## 11. Feature Analysis & Visualization

In [ ]:
# Separate features by class
benign_feat = features_matrix[labels == 0]
malignant_feat = features_matrix[labels == 1]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Differential Equation Feature Analysis', fontsize=15, fontweight='bold')

# 1. ODE coefficients distribution
coeff_names = ['\u03b1 (alpha)', '\u03b2 (beta)', '\u03b3 (gamma)']
for i, (name, ax_idx) in enumerate(zip(coeff_names, [axes[0, 0], axes[0, 1], axes[0, 2]])):
    ax_idx.hist(benign_feat[:, i], bins=30, alpha=0.6, color='seagreen', label='Benign', density=True)
    ax_idx.hist(malignant_feat[:, i], bins=30, alpha=0.6, color='crimson', label='Malignant', density=True)
    ax_idx.set_title(f'ODE Coefficient {name}', fontsize=12, fontweight='bold')
    ax_idx.set_xlabel('Value')
    ax_idx.set_ylabel('Density')
    ax_idx.legend(fontsize=9)
    ax_idx.grid(True, alpha=0.3)

# 2. RMSE distributions
rmse_indices = [-7, -6, -5, -4]  # euler_ode, rk4_ode, euler_ide, rk4_ide RMSE
rmse_names = ['Euler ODE', 'RK4 ODE', 'Euler IDE', 'RK4 IDE']
rmse_colors = ['darkorange', 'forestgreen', 'mediumpurple', 'crimson']

for i, (idx, name, color) in enumerate(zip(rmse_indices[:3], rmse_names[:3], rmse_colors[:3])):
    axes[1, i].hist(benign_feat[:, idx], bins=30, alpha=0.6, color='seagreen', label='Benign', density=True)
    axes[1, i].hist(malignant_feat[:, idx], bins=30, alpha=0.6, color='crimson', label='Malignant', density=True)
    axes[1, i].set_title(f'{name} RMSE', fontsize=12, fontweight='bold')
    axes[1, i].set_xlabel('RMSE')
    axes[1, i].set_ylabel('Density')
    axes[1, i].legend(fontsize=9)
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'de_feature_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare average trajectories: Benign vs Malignant
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
t = np.arange(10)

traj_ranges = {
    'Euler ODE Traj': (3, 13),
    'RK4 ODE Traj': (13, 23),
    'Euler IDE Traj': (43, 53),
    'RK4 IDE Traj': (53, 63),
}

for idx, (name, (start, end)) in enumerate(traj_ranges.items()):
    mean_b = np.mean(benign_feat[:, start:end], axis=0)
    std_b = np.std(benign_feat[:, start:end], axis=0)
    mean_m = np.mean(malignant_feat[:, start:end], axis=0)
    std_m = np.std(malignant_feat[:, start:end], axis=0)
    
    axes[idx].plot(t, mean_b, 'o-', color='seagreen', linewidth=2.5, label='Benign', markersize=7)
    axes[idx].fill_between(t, mean_b - std_b, mean_b + std_b, alpha=0.15, color='seagreen')
    axes[idx].plot(t, mean_m, 's-', color='crimson', linewidth=2.5, label='Malignant', markersize=7)
    axes[idx].fill_between(t, mean_m - std_m, mean_m + std_m, alpha=0.15, color='crimson')
    
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('t')
    axes[idx].set_ylabel('Value')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

fig.suptitle('Mean DE/IDE Trajectories: Benign vs Malignant', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 12. Feature Normalization

Normalize features using **StandardScaler** (zero mean, unit variance) for optimal CNN performance.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_normalized = scaler.fit_transform(features_matrix)

print(f"Feature normalization applied:")
print(f"   Before: mean={features_matrix.mean():.4f}, std={features_matrix.std():.4f}")
print(f"   After:  mean={features_normalized.mean():.6f}, std={features_normalized.std():.4f}")
print(f"   Shape: {features_normalized.shape}")

## 13. Save DE Features

Save the extracted and normalized features for the CNN classification notebook.

In [ ]:
# Save features
features_path = os.path.join(OUTPUT_DIR, 'de_features.npz')

np.savez_compressed(
    features_path,
    features=features_matrix,
    features_normalized=features_normalized,
    labels=labels,
    feature_names=np.array(feature_names),
    scaler_mean=scaler.mean_,
    scaler_scale=scaler.scale_
)

# Also save as CSV for inspection
feat_df = pd.DataFrame(features_matrix, columns=feature_names)
feat_df['label'] = labels
feat_df.to_csv(os.path.join(OUTPUT_DIR, 'de_features.csv'), index=False)

print("✅ DE features saved successfully!")
print(f"   NPZ file : {features_path} ({os.path.getsize(features_path)/1024:.1f} KB)")
print(f"   CSV file : {os.path.join(OUTPUT_DIR, 'de_features.csv')}")
print(f"   Features : {features_matrix.shape[1]} per sample")
print(f"   Samples  : {features_matrix.shape[0]}")

---

## ✅ Summary

In this notebook, we implemented differential equation modeling for brain tumor classification:

### Methods Implemented

| Method | Type | Order | Key Advantage |
|--------|------|-------|---------------|
| Euler ODE | DE | 1st | Simple, fast computation |
| RK4 ODE | DE | 4th | Higher accuracy, better stability |
| Euler IDE | IDE | 1st | Captures temporal memory effects |
| RK4 IDE | IDE | 4th | Best accuracy with memory modeling |

### Feature Vector (90 features per sample)
- 3 ODE coefficients (α, β, γ)
- 40 trajectory values (4 methods × 10 time points)
- 40 residual values (4 methods × 10 time points)
- 7 statistical summaries (RMSE, energy, slope, curvature)

### ➡️ Next: [Notebook 04 - CNN Classification Model](./04_CNN_Classification_Model.ipynb)

The extracted features will be used to train a 1D Convolutional Neural Network for binary classification (Benign vs Malignant).